### Starter pack for the Brain age prediction EEG data set (Helsinki 1K-Kids) : Calculation of simple EEG features

**Last updated:** 31.10.2026

This reads pre-processed EEG files and computes two sets of features, Hjorth parameters and Band power.

In [ ]:
%load_ext autoreload
%autoreload 2

### Configuration

In [ ]:
import os
from pathlib import Path

path = Path('data')
edf_path = path / Path('edf_files')
pre_processed_path = path / Path('pre_processed')

TARGET_DURATION = 901 # Most files (but not all) are 901 seconds

# Downsample from whatever sampling rate used during recording to EEG
# to resampling_rate
RESAMPLING_RATE = 64

# Amplitude cutoff (to avoid overflow/roundoff errors)
MAX_AMPLITUDE = 2000 # microvolt

# Filter settings for EEG pre-processing
LOW_CUT = 0.5
HIGH_CUT = 30.0
NOTCH = 50.0 # The voltage in Europe is 50 Hz

# EEG preprocessing - montage derivation and filter, and feature calculation
montage = ['FP1-F7', 'F7-T3', 'T3-T5', 'T5-O1', 'FP2-F8', 'F8-T4',
           'T4-T6',  'T6-O2', 'T3-C3', 'C3-CZ',  'CZ-C4', # 'A1-T3'
           'C4-T4',  'FP1-F3','F3-C3', 'C3-P3',  'P3-O1', # 'T4-A2'
           'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2']
MONTAGE = {i : chans for i, chans in enumerate(montage)}

### Bare-bones EEG viewer

In [ ]:
import re
import torch
import matplotlib.pyplot as plt
import numpy as np

patient = 'U4S455'
t_start = 130 # Offset from beginning of recording (seconds)
t_dur = 10    # On-screen duration

yoffset = 0
data_filtered = torch.load(pre_processed_path / (patient + '.pt'))
plt.figure(figsize=(12,6))
nsamples = data_filtered.shape[1]
t = np.arange(RESAMPLING_RATE*TARGET_DURATION) / RESAMPLING_RATE
istart = int(t_start * RESAMPLING_RATE)
iend = int((t_start + t_dur)* RESAMPLING_RATE)
for ichan in range(data_filtered.shape[0]):
    col = 'tab:blue' if bool(re.search(r'[02468]', montage[ichan])) else 'tab:orange'
    plt.plot(t[istart:iend], data_filtered[ichan,istart:iend] + yoffset,linewidth=0.5, color=col)
    plt.text(t[istart]-0.5, yoffset, montage[ichan])
    yoffset += 150
plt.title(f'{patient} (start={t_start}, end={t_start + t_dur})')
plt.xlabel('Offset from start of recording (seconds)')
plt.show()

### Age distribution in the dataset

In [ ]:
import pandas as pd

df = pd.read_csv(path / 'anon_codex_age_sex.csv')
age_mapping = {s[0] : s[1] for s in list(df.itertuples(index=False, name=None))} # filename -> age

In [ ]:
plt.hist([value for key, value in age_mapping.items()], bins=20)
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

### Basic feature extraction

Compute "Hjorth" features using a sliding window from `t_start` `t_start + t_dur` of 1 second duration. Channels are processed independently.

In [ ]:
import torch

# Example calculation of a simple EEG feature based on the signal amplitude

patient = 'U4S455'
t_start = 0 # Starting point of segment (seconds)
t_dur = 10  # How far to go from t_start (seconds)

data = torch.load(pre_processed_path / (patient + '.pt'))

def calc_hjorth1(x):
    # Hjorth activity parameter (= variance)
    return torch.var(x)

def calc_hjorth2(x):
    # Input: x contains EEG data from a single channel
    # Output: Hjorth mobility parameter
    return torch.var(torch.diff(x)) / (torch.var(x) + 1e-6).item()

# Compute feature for a sliding window
feature = np.zeros((data.shape[0], t_dur))
for chan in range(data.shape[0]):
    for t in range(t_start, t_start + t_dur):
        feature[chan,t - t_start] = calc_hjorth1(data[chan, np.arange(t * RESAMPLING_RATE, \
                                                                      (t+1) * RESAMPLING_RATE)])
plt.figure(figsize=(12,6))
y_scale = 0.5
for chan in range(18):
    plt.plot(np.arange(t_start, t_start + t_dur), feature[chan,:]*y_scale + chan*10)
    plt.text(t_start-0.5, feature[chan,0]*y_scale + chan*10, montage[chan])
plt.title('Hjorth 2 - sliding window computation')
plt.xlabel('Seconds')
plt.ylabel('Feature value')
plt.show()

### Attempt to recreate Figure 4A in the EEG dataset paper.

Compute the Hjorth 2 (mobility) for all the files in the dataset.

Not sure about the exact procedure used in the paper, this is a quick and dirty attempt at replication.

In [ ]:
t_start = 0    # Starting point of segment (seconds) - pre-N2 onset region
t_window = 300 # Amount of EEG used to compute a single feature value

patient_h2 = []
for patient in age_mapping.keys():
    data = torch.load(pre_processed_path / (patient.replace('.edf', '.pt')))

    # Compute single feature value for each channel
    feature = np.zeros((data.shape[0]))
    for chan in range(data.shape[0]):
        feature[chan] = calc_hjorth2(data[chan, np.arange(t_start * RESAMPLING_RATE, \
                                                         (t_start + t_window) * RESAMPLING_RATE)])
    patient_h2.append((age_mapping[patient], np.nanmedian(feature).item())) # Use median to get robust estimate, skip nans

In [ ]:
age, feature_val = zip(*patient_h2)
plt.scatter(list(age), list(feature_val),s=5)
plt.xlabel('Age')
plt.ylabel('Hjorth 2')
plt.title('Pre-N2 onset' if t_start < 300 else 'Post-N2 onset')
plt.show()

print(f'Correlation coefficient: r = {np.corrcoef(list(age), list(feature_val))[0,1]:.3f}')

We could construct a baseline model by performing linear regression model on Hjorth-2 feature values in the training set.

In the paper they get r=0.849 which is considerably higher but they remove corrupt recordings/segments using "artifact detection" algorithms.

Band power - Power in delta (0.5 - 4 Hz), theta (4 - 8 Hz), alpha (8 - 12 Hz) and beta (12 and up) bands are classical EEG features.

Compute features using a sliding window from `t_start` `t_start + t_dur` of 1 second duration. For use in classification/regression models one uses several second duration. Channels are processed independently.

In [ ]:
from scipy.integrate import simpson
from scipy import signal

def band_power(x, fs, low=0.5, high=4, win=1):
    # Calculate the power spectrum of a signal
    # Input: x contains EEG data from a single channel
    #        fs sampling rate
    #        low, high are the frequency band limits
    #        win is the window length
    # Output: Absolute and relative power in the [low,high] band

    # Code is based on: https://raphaelvallat.com/bandpower.html

    # Welch's averaged periodogram is based on the Fast Fourier Transform (FFT)
    freqs, psd = signal.welch(x, fs, nperseg=win*fs)
    idx_delta = np.logical_and(freqs >= low, freqs <= high)
    freq_res = freqs[1] - freqs[0]

    # Compute the absolute power by approximating the area under the curve
    abs_power = simpson(psd[idx_delta], dx=freq_res)

    # Relative delta power (expressed as a percentage of total power)
    total_power = simpson(psd, dx=freq_res)
    rel_power = abs_power / total_power
    return abs_power, rel_power

low, high = 8, 12 # Alpha band (Hz)
psd_feature = np.zeros((data.shape[0], t_dur))
for chan in range(data.shape[0]):
    for t in range(t_start, t_start + t_dur):
        psd_feature[chan,t - t_start] = band_power(data[chan,np.arange(t * RESAMPLING_RATE, \
                                                                       (t+1) * RESAMPLING_RATE)],\
                                                   RESAMPLING_RATE, low, high)[0] # Absolute power

plt.figure(figsize=(12,6))
for chan in range(data.shape[0]):
    plt.plot(np.arange(t_start, t_start + t_dur), psd_feature[chan,:]*y_scale + chan*20)
    plt.text(t_start-0.5, psd_feature[chan,0]*y_scale + chan*20, montage[chan])
plt.title('Band power')
plt.show()